# Aggregate Contig Taxonomic Annotations
Lucia Winkler 29.07.2025

## What this notebook does:
This notebook takes the mmseqs output folder of nf-mmseqs and the pydamage_results.csv from the nf-coreMAG/Ancient_DNA output. Full taxonomic lineage information is added with taxopy. From the pydamage information on mapped reads and contig length I'm calculating coverages using the formula for TPM (knowing that the reads are not transcripts)



### Packages

In [1]:
import pandas as pd
import os
import glob
import taxopy
import re
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import pysam
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform, jaccard
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from plotnine import *
from scipy.stats import mannwhitneyu


### Database

In [2]:
taxdb = taxopy.TaxDb()

### Functions

In [ ]:
"Read and concatenate contig taxonomic annotation"
# Configuration
DATA_PATH = '<path to>/MMSeqs_Contig_Taxonomy'
TSV_PATTERN = os.path.join(DATA_PATH, '*.tsv')

# Columns of interest
COLUMN_MAP = {
    0: 'contig',
    1: 'taxID',
    2: 'rank',
    3: 'taxon'
}

def load_and_process_tsv(file_path):
    sample_name = os.path.splitext(os.path.basename(file_path))[0]
    df = pd.read_csv(file_path, sep='\t', header=None)
    df = df.assign(sample=sample_name)

    # Rename selected columns
    df = df.rename(columns=COLUMN_MAP)

    # Drop columns 0 to 8 (inclusive), except renamed ones
    keep_cols = list(COLUMN_MAP.values()) + ['sample']
    df = df[keep_cols]
    return df

In [4]:
"Add rank names to tax. annotation"


def get_genus(taxid, taxdb=taxdb):
    try:
        _ = taxopy.Taxon(taxid, taxdb)
        if "genus" in _.rank_name_dictionary:
            g =  _.rank_name_dictionary['genus']
        else:
            g = None
        if "species" in _.rank_name_dictionary:
            s =  _.rank_name_dictionary['species']
        else:
            s = None
    except taxopy.exceptions.TaxidError:
        g = None
        s = None
    return g, s

In [5]:
def get_taxonomic_ranks(taxid, taxdb=taxdb):
    ranks = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
    try:
        taxon = taxopy.Taxon(taxid, taxdb)
        rank_values = []
        for rank in ranks:
            rank_values.append(taxon.rank_name_dictionary.get(rank, None))
    except taxopy.exceptions.TaxidError:
        rank_values = [None] * len(ranks)
    return tuple(rank_values)

In [ ]:
'read unfiltered pydamage results table'
# Configuration
BASE_DIR = Path('<path to>/Pydamage_Full/analyze')
RESULT_FILENAME = 'pydamage_results/pydamage_results.csv'

# Patterns to support both TDM and EXB formats
ASSEMBLY_REGEX = r"(TDM\d{3}|EXB\d{3}_A\d{4})"

def find_pydamage_files(base_dir: Path, result_filename: str):
    """Finds all pydamage result files in subfolders."""
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    result_paths = [(f.name, f / result_filename) for f in folders if (f / result_filename).exists()]
    return result_paths

def load_and_annotate(file_info):
    """Loads a CSV and adds 'sample' column."""
    name, path = file_info
    df = pd.read_csv(path, usecols=range(16))
    df['sample'] = name
    df = df.rename(columns={'reference': 'contig'})
    return df

def standardize_assembly_name(name):
    """Extracts TDMxxx or EXB080_Axxxx pattern."""
    match = re.search(ASSEMBLY_REGEX, name)
    return match.group(0) if match else name  # fallback to original name if no match

### Load data

In [7]:
# Process all taxonomic annotation TSV files and concatenate into df
all_dfs = [load_and_process_tsv(f) for f in glob.glob(TSV_PATTERN)]
tax_df = pd.concat(all_dfs, ignore_index=True)


In [8]:
# add genus and species name to df
tax_df[['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']] = pd.DataFrame(
    tax_df['taxID'].map(get_taxonomic_ranks).tolist(),
    index=tax_df.index
)

In [9]:
tax_df

,contig,taxID,rank,taxon,sample,kingdom,phylum,class,order,family,genus,species
0,k99_4854,2,superkingdom,Bacteria,TDM056,None,None,None,None,None,None,None
1,k99_11661,33809,no rank,unclassified Betaproteobacteria,TDM056,Pseudomonadati,Pseudomonadota,Betaproteobacteria,None,None,None,None
2,k99_4867,2935764,species,Nitrospiraceae bacterium AH_259_D15_M11_P09,TDM056,Pseudomonadati,Nitrospirota,Nitrospiria,Nitrospirales,Nitrospiraceae,None,Nitrospiraceae bacterium AH_259_D15_M11_P09
3,k99_1310,356,order,Hyphomicrobiales,TDM056,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Hyphomicrobiales,None,None,None
4,k99_4877,1298915,no rank,unclassified Nitrospirota,TDM056,Pseudomonadati,Nitrospirota,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
2073592,k99_19008,2,superkingdom,Bacteria,TDM040,None,None,None,None,None,None,None
2073593,k99_10976,1891241,species,Betaproteobacteria bacterium,TDM040,Pseudomonadati,Pseudomonadota,Betaproteobacteria,None,None,None,Betaproteobacteria bacterium
2073594,k99_8249,1234,genus,Nitrospira,TDM040,Pseudomonadati,Nitrospirota,Nitrospiria,Nitrospirales,Nitrospiraceae,Nitrospira,None
2073595,k99_19018,33809,no rank,unclassified Betaproteobacteria,TDM040,Pseudomonadati,Pseudomonadota,Betaproteobacteria,None,None,None,None


In [10]:
# Load pydamage data
# Step 1: Locate files
file_list = find_pydamage_files(BASE_DIR, RESULT_FILENAME)

# Step 2: Load and annotate
dataframes = [load_and_annotate(info) for info in file_list]

# Step 3: Concatenate into single DataFrame
pydamage_df = pd.concat(dataframes, ignore_index=True)

# Step 4: Standardize assembly names
pydamage_df['sample'] = pydamage_df['sample'].apply(standardize_assembly_name)

In [ ]:
#Load age model
age_model = pd.read_csv("<path to>/summarized_age_model.csv", sep=';', decimal=',')
age_model['sample'] = age_model['sample_name'].str.replace("_", "")

In [12]:
pydamage_df['sample'].unique()

array(['EXB080_A0101', 'TDM033', 'TDM042', 'EXB080_A0901', 'EXB080_A0801',
       'TDM056', 'TDM030', 'TDM001', 'TDM038', 'TDM051', 'TDM017',
       'TDM019', 'TDM055', 'TDM036', 'TDM003', 'TDM037', 'TDM027',
       'EXB080_A1101', 'TDM013', 'TDM053', 'TDM052', 'TDM060', 'TDM049',
       'TDM005', 'TDM043', 'TDM014', 'TDM023', 'TDM015', 'TDM054',
       'TDM024', 'EXB080_A1001', 'TDM007', 'TDM034', 'TDM016', 'TDM008',
       'TDM012', 'EXB080_A1201', 'TDM022', 'TDM035', 'TDM006', 'TDM057',
       'TDM010', 'TDM045', 'TDM028', 'TDM025', 'TDM046', 'TDM020',
       'TDM061', 'EXB080_A0102', 'TDM011', 'TDM039', 'TDM031', 'TDM002',
       'TDM029', 'TDM004', 'TDM021', 'EXB080_A1401', 'EXB080_A1301',
       'TDM040', 'TDM044', 'TDM048', 'TDM018', 'TDM026', 'TDM041',
       'TDM032', 'TDM059', 'TDM047', 'TDM050', 'TDM058'], dtype=object)

In [13]:
full_df = pydamage_df.merge(tax_df, on=["sample", "contig"] , how="left")
full_df = full_df.merge(age_model, on= ['sample'], how='left')


In [14]:
tax_df['sample'].unique()

array(['TDM056', 'TDM025', 'TDM053', 'TDM061', 'TDM051', 'TDM019',
       'EXB080_A1101', 'TDM029', 'EXB080_A0801', 'TDM010', 'TDM016',
       'TDM054', 'TDM001', 'TDM038', 'TDM052', 'TDM020', 'TDM013',
       'TDM055', 'TDM044', 'EXB080_A1001', 'TDM050', 'TDM031', 'TDM039',
       'EXB080_A0901', 'TDM046', 'EXB080_A0101', 'TDM006', 'TDM060',
       'TDM002', 'TDM030', 'TDM035', 'TDM058', 'EXB080_A1201', 'TDM015',
       'TDM005', 'TDM018', 'TDM017', 'TDM059', 'TDM003', 'TDM012',
       'TDM008', 'EXB080_A1401', 'TDM022', 'TDM028', 'TDM027', 'TDM026',
       'TDM049', 'TDM045', 'TDM048', 'TDM036', 'TDM021', 'TDM037',
       'TDM033', 'TDM047', 'EXB080_A1301', 'EXB080_A0102', 'TDM023',
       'TDM024', 'TDM004', 'TDM043', 'TDM011', 'TDM041', 'TDM032',
       'TDM014', 'TDM042', 'TDM057', 'TDM007', 'TDM034', 'TDM040'],
      dtype=object)

In [15]:
# drop some columns I don't need here
working_df=  full_df.drop(columns=[ 'predicted_accuracy', 'null_model_p0', 'null_model_p0_stdev','damage_model_p', 
                                'damage_model_p_stdev', 'damage_model_pmin','damage_model_pmin_stdev', 'damage_model_pmax',
                                'damage_model_pmax_stdev', 'pvalue', 'qvalue', 'RMSE', 'sample_name'])

In [16]:
cols_to_fix = ['taxon','kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']  

working_df[cols_to_fix] = working_df[cols_to_fix].where(pd.notnull(working_df[cols_to_fix]), None)

In [17]:
working_df

,contig,nb_reads_aligned,coverage,reflen,sample,taxID,rank,taxon,kingdom,phylum,class,order,family,genus,species,age_mean,age_sd,lower_layer_depth (cm)
0,k59_143,110,19.472,284,EXB080_A0101,32008.0,genus,Burkholderia,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Burkholderiaceae,Burkholderia,None,NaN,NaN,NaN
1,k59_40,116,21.008,238,EXB080_A0101,1224.0,phylum,Pseudomonadota,Pseudomonadati,Pseudomonadota,None,None,None,None,None,NaN,NaN,NaN
2,k59_450,80,17.905,222,EXB080_A0101,119060.0,family,Burkholderiaceae,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Burkholderiaceae,None,None,NaN,NaN,NaN
3,k59_664,134,22.769,255,EXB080_A0101,1335308.0,species,Burkholderia sp. AU4i,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Burkholderiaceae,Burkholderia,Burkholderia sp. AU4i,NaN,NaN,NaN
4,k59_430,154,28.060,250,EXB080_A0101,1224.0,phylum,Pseudomonadota,Pseudomonadati,Pseudomonadota,None,None,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2302798,k99_25811,122,20.064,327,TDM058,1747.0,species,Cutibacterium acnes,Bacillati,Actinomycetota,Actinomycetes,Propionibacteriales,Propionibacteriaceae,Cutibacterium,Cutibacterium acnes,34793.6,273.56561,58.0
2302799,k99_24840,78,16.465,303,TDM058,NaN,NaN,None,None,None,None,None,None,None,None,34793.6,273.56561,58.0
2302800,k99_11950,122,16.934,457,TDM058,1783272.0,kingdom,Bacillati,Bacillati,None,None,None,None,None,None,34793.6,273.56561,58.0
2302801,k99_21110,76,15.616,320,TDM058,2.0,superkingdom,Bacteria,None,None,None,None,None,None,None,34793.6,273.56561,58.0


### Calculate contig coverages (TPM)

In [18]:
 # Step 1: Calculate RPK for each contig
working_df["rpk"] = working_df["nb_reads_aligned"] / (working_df["reflen"] / 1000)

# Step 2: Calculate TPM per sample
# Group by sample, calculate per-sample sum of RPKs
working_df["rpk_sum"] = working_df.groupby("sample")["rpk"].transform("sum")

# Step 3: Compute TPM
working_df["tpm"] = (working_df["rpk"] / working_df["rpk_sum"]) * 1e6


## Save working DF for use in R

In [ ]:
working_df.to_csv("contig_tax_tpm.csv", sep=";", index=False, encoding="utf-8", header=True)